# 01 - PySpark DataFrame Basics

Objetivo: ler um CSV, inspecionar o schema, filtrar linhas e agregar dados.

In [3]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

java_home = Path("/usr/local/opt/openjdk@17")
if not java_home.exists():
    java_home = Path("/opt/homebrew/opt/openjdk@17")

os.environ["JAVA_HOME"] = str(java_home)
os.environ["PATH"] = f"{java_home / 'bin'}:{os.environ['PATH']}"
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")

print(f"Python: {sys.executable}")
print(f"JAVA_HOME: {os.environ['JAVA_HOME']}")

Python: /usr/local/bin/python3
JAVA_HOME: /usr/local/opt/openjdk@17


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date

In [ ]:
spark = (
    SparkSession.builder.appName("notebook-01-dataframe-basics")
    .master("local[*]")
    .getOrCreate()
)


## Ler dados

Em Databricks, o `spark` normalmente ja existe. Localmente, criamos a `SparkSession` acima.

In [ ]:
orders = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .csv(str(PROJECT_ROOT / "data/raw/orders.csv"))
)

orders.printSchema()
orders.show(truncate=False)

## Transformar colunas

`withColumn` cria ou substitui uma coluna. Aqui garantimos tipos mais corretos.

In [ ]:
clean_orders = (
    orders.withColumn("order_date", to_date(col("order_date")))
    .withColumn("quantity", col("quantity").cast("int"))
)

clean_orders.printSchema()

## Filtrar e selecionar

Esta e uma operacao classica de Silver layer: manter apenas linhas validas/uteis.

In [ ]:
delivered_orders = clean_orders.filter(col("status") == "delivered")

delivered_orders.select(
    "order_id", "customer_id", "product_id", "order_date", "quantity"
).show(truncate=False)

## Agregar

`groupBy(...).count()` e uma das primeiras formas de validar distribuicao dos dados.

In [ ]:
clean_orders.groupBy("status").count().orderBy("status").show(truncate=False)

## Para praticar

Tenta alterar as celulas para responder:

1. Quantas encomendas existem por cliente?
2. Quais encomendas têm `quantity >= 2`?
3. Quantas encomendas foram canceladas ou devolvidas?